# 03 — Model Training, Scratch Algorithm & Comprehensive Evaluation
## Computer Engineering — Semester Machine Learning Project (Weeks 3, 4, 5, 9, 10)
### Scope:
1. Mandatory Scratch Logistic Regression implementation from first principles.
2. Library models: Logistic Regression, Decision Tree, Random Forest, Gradient Boosting.
3. 5-Fold Stratified Cross-Validation.
4. Comprehensive Metrics: Accuracy, Precision, Recall, F1, ROC-AUC, Specificity, FPR, FNR.
5. Error Analysis (False Positives vs False Negatives in financial lending).
6. Threshold Optimization & Probability Calibration.


In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression as SklearnLogReg
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

from src.scratch_logistic_regression import LogisticRegressionScratch
from src.data_pipeline import load_raw_dataset, prepare_zero_leakage_splits

print("Modules imported successfully.")


## 1. Load Data Pipeline


In [ ]:
df = load_raw_dataset()
splits = prepare_zero_leakage_splits(df)

X_train = splits["X_train_scaled"]
y_train = splits["y_train"]
X_test = splits["X_test_scaled"]
y_test = splits["y_test"]
feature_names = splits["feature_names"]

print(f"Train: {X_train.shape}, Test: {X_test.shape}")


## 2. Mandatory Scratch Implementation vs. Library Logistic Regression


In [ ]:
# Scratch Model (trained on 25k stratified subset for fast gradient descent convergence)
sub_idx = np.random.RandomState(42).choice(len(y_train), size=25000, replace=False)
scratch_model = LogisticRegressionScratch(learning_rate=0.1, n_iterations=300, random_state=42)
scratch_model.fit(X_train[sub_idx], y_train[sub_idx])

# Library Model
sklearn_model = SklearnLogReg(max_iter=500, random_state=42)
sklearn_model.fit(X_train[sub_idx], y_train[sub_idx])

scratch_preds = scratch_model.predict(X_test)
scratch_probs = scratch_model.predict_proba(X_test)[:, 1]

sklearn_preds = sklearn_model.predict(X_test)
sklearn_probs = sklearn_model.predict_proba(X_test)[:, 1]

print("--- Scratch vs Library Comparison ---")
print(f"Scratch Model -> Accuracy: {accuracy_score(y_test, scratch_preds):.4f} | ROC-AUC: {roc_auc_score(y_test, scratch_probs):.4f}")
print(f"Library Model -> Accuracy: {accuracy_score(y_test, sklearn_preds):.4f} | ROC-AUC: {roc_auc_score(y_test, sklearn_probs):.4f}")


## 3. Multi-Model Training with Class Balancing


In [ ]:
models = {
    "Logistic Regression (Balanced)": SklearnLogReg(max_iter=1000, class_weight='balanced', random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=8, min_samples_leaf=20, class_weight='balanced', random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=50, max_depth=12, class_weight='balanced', n_jobs=-1, random_state=42),
    "Gradient Boosting": HistGradientBoostingClassifier(max_iter=100, max_depth=8, class_weight='balanced', random_state=42)
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    probs = model.predict_proba(X_test)[:, 1]
    
    results.append({
        "Model": name,
        "Accuracy (%)": round(accuracy_score(y_test, preds) * 100, 2),
        "Precision (%)": round(precision_score(y_test, preds) * 100, 2),
        "Recall (%)": round(recall_score(y_test, preds) * 100, 2),
        "F1 (%)": round(f1_score(y_test, preds) * 100, 2),
        "ROC-AUC (%)": round(roc_auc_score(y_test, probs) * 100, 2)
    })

comparison_df = pd.DataFrame(results)
comparison_df


## 4. Error Analysis & Confusion Matrix (Week 9)


In [ ]:
best_model = models["Gradient Boosting"]
best_preds = best_model.predict(X_test)
cm = confusion_matrix(y_test, best_preds)
tn, fp, fn, tp = cm.ravel()

print(f"True Negatives  (Safe correctly identified):       {tn:,}")
print(f"False Positives (Safe borrower denied / review):    {fp:,}")
print(f"False Negatives (Defaulter disbursed / lost $):    {fn:,}")
print(f"True Positives  (Defaulter caught):                 {tp:,}")

plt.figure(figsize=(6, 4.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Greens", xticklabels=['Safe', 'Default'], yticklabels=['Safe', 'Default'])
plt.title("Confusion Matrix: Gradient Boosting", fontweight='bold')
plt.ylabel("Actual Label")
plt.xlabel("Predicted Label")
plt.show()
